RAG FROM SCRATCH

we are going to build NitriChat to "Chat with a nutrition textbook."<br>
Specially;<br> <br>
1.Open a pdf document<br>
2.Format the text of the PDF textbook ready for the embedding model.<br>
3.Embeded all of the chunks of text in the textbook and turn them into numerical representation(embedding)which can store for later.<br>
4.Build a retrieval system that uses vector search to find a relevant chunk of text based on a query.<br>
5.Create a prompt that incorporates the retrieved pieces of text.<br>
6.Generate an answer to a query based on the passage of the notebook with LLM.

<br> Summary!<br>
1.Steps 1-3: Document preprocessing and embedding creation.<br>
2.Steps 4-6: Search and answer.<br>
3.Embed text chunks with embedding model.<br>
4.Save embeddings to file for later(embeddings will store on file for many years or until you lose your hard drive.)


# Import PDF Document

In [1]:
# import requests
# import os

# pdf_path = "human-nutrition-pdf"

# #Download pdf and save
# if not os.path.exists(pdf_path):
#   print("Downlaoding file....")
#   url = ""
#   file_name = pdf_path
#   response = requests.get(url)
#   #check download succeed
#   if response.status_code == 200:
#     with open(file_name, 'wb')as f:
#       f.write(response.content)
#       print(f"download complete and saved to {file_name}")
#   else:
#     print(f"Download failed. status code:{response.status_code} ")

# else:
#     print(f"file already exits {pdf_path}")



In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 68.9 MB/s eta 0:00:00


In [3]:
import pymupdf
import re

pdf_path = "/content/human-nutrition.pdf"
def text_formatted(text: str) -> str:
  return re.sub(r'\s+', ' ', text).strip()  # \n मात्र नभई सबै extra spaces पनि हटाउँछ
  return cleaned_text

def extract_text_from_pdf(pdf_path) -> list[dict]:
    doc = pymupdf.open(pdf_path)

    pages_and_text = []
    for page_number,page in enumerate(doc):
      text = page.get_text()
      text = text_formatted(text)
      pages_and_text.append({"page_number":page_number-19,
                             "page_character_count":len(text),
                             "page_word_count":len(text.split()),
                             "page_sentence_count":len(text.split(".")),
                             "text":text})
    return pages_and_text

pages_and_text = extract_text_from_pdf(pdf_path=pdf_path)
pages_and_text[20]

{'page_number': 1,
 'page_character_count': 66,
 'page_word_count': 12,
 'page_sentence_count': 1,
 'text': 'PART I BASIC CONCEPTS IN NUTRITION BASIC CONCEPTS IN NUTRITION | 1'}

In [4]:
import random
random.sample(pages_and_text,k=2)

[{'page_number': 512,
  'page_character_count': 3085,
  'page_word_count': 447,
  'page_sentence_count': 30,
  'text': 'pregnant women in the world have iron-deficiency anemia. 3 The main causes of iron deficiency worldwide are parasitic worm infections in the gut, causing excessive blood loss, and malaria, a parasitic disease that destroys red blood cells. In the developed world, iron deficiency is more the result of dietary insufficiency and/or excessive blood loss during menstruation or childbirth. At-Risk Populations Infants, children, adolescents, and women are the populations most at risk worldwide for iron-deficiency anemia by all causes. Infants, children, and even teens require more iron because iron is essential for growth. In these populations, iron deficiency (and eventually iron-deficiency anemia) can also cause the following signs and symptoms: poor growth, failure to thrive, and poor performance in school, as well as mental, motor, and behavioral disorders. Women who exp

In [5]:
#convert to dataframe
import pandas as pd

df = pd.DataFrame(pages_and_text)
df.head(5)

,page_number,page_character_count,page_word_count,page_sentence_count,text
0,-19,15,2,1,Human Nutrition
1,-18,0,0,1,
2,-17,105,14,1,HUMAN NUTRITION Louisiana Edition MELISSA JOHN...
3,-16,238,30,3,Human Nutrition Copyright © 2022 by Melissa Jo...
4,-15,518,79,3,CONTENTS Preface xiii About the Contributors x...


Further Preprocessing the Text => Spliting Pages to Sentences.

Two ways:<br>
1.By using split"."<br>
2.By using NLP library such as Spacy and NLTP.We gonna use spacy for now.

In [6]:
!pip install spacy

In [7]:
import spacy

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

#Demo
text = "This is sen one.This is sen two."
doc = nlp(text)

list(doc.sents)

[This is sen one., This is sen two.]

In [8]:
# sentence = []
# for pages in pages_and_text:
#   doc = nlp(pages["text"])
#   sentence.extend(sent.text.strip() for sent in doc.sents)

# sentence[1]

In [9]:
#optional approach be like and good approach
for items in pages_and_text:

  docs = nlp(items["text"])
  items["sentence"] = [str(sentence).strip() for sentence in docs.sents]

  items["sentence_count"] = len(items["sentence"])

pages_and_text[25]

{'page_number': 6,
 'page_character_count': 1846,
 'page_word_count': 281,
 'page_sentence_count': 15,
 'text': 'Lipids Lipids are also a family of molecules composed of carbon, hydrogen, and oxygen, but unlike carbohydrates, they are insoluble in water. Lipids are found predominantly in butter, oils, meats, dairy products, nuts, and seeds, and in many processed foods. The three main types of lipids are triglycerides (triacylglycerols), phospholipids, and sterols. The main job of lipids is to provide or store energy. Lipids provide more energy per gram than carbohydrates (nine kilocalories per gram of lipids versus four kilocalories per gram of carbohydrates). In addition to energy storage, lipids serve as a major component of cell membranes, surround and protect organs (in fat-storing tissues), provide insulation to aid in temperature regulation, and regulate many other functions in the body. Proteins Proteins are macromolecules composed of chains of subunits called amino acids. Amino

In [10]:
df = pd.DataFrame(pages_and_text)
df.describe().round(1)

,page_number,page_character_count,page_word_count,page_sentence_count,sentence_count
count,906.0,906.0,906.0,906.0,906.0
mean,433.5,1488.1,226.3,19.1,14.3
std,261.7,935.4,143.1,15.4,10.6
min,-19.0,0.0,0.0,1.0,0.0
25%,207.2,725.5,111.2,8.0,5.0
50%,433.5,1418.0,218.0,16.0,12.0
75%,659.8,2207.5,331.8,28.0,21.0
max,886.0,3670.0,598.0,110.0,59.0


Now,lets further chunks the sentences into the smaller chunks.i.e into 10 sentences.

Why chunking?<br>
1.Our text are easier to filter.<br>
2.Our chunk can be fit into our embedding model.<br>
3.Our context passed to LLM can be more specific and focused.


We can chunk sentence using Langchain but now we will gonna use python.

In [11]:
pages_and_text[24]

{'page_number': 5,
 'page_character_count': 2975,
 'page_word_count': 459,
 'page_sentence_count': 28,
 'text': 'Macronutrients Nutrients that are needed in large amounts are called macronutrients. There are three classes of macronutrients: carbohydrates, lipids, and proteins. Through various biological processes occurring in the body, these nutrients can be metabolized to produce cellular energy. The energy from macronutrients comes from their chemical bonds. This chemical energy is converted into cellular energy that is then utilized to perform work, allowing our bodies to conduct their basic functions. The energy obtained from food, which thus becomes cellular energy, is measured in calorie units. A unit of measurement of food energy is the calorie. On nutrition food labels, the amount given for “calories” is actually equivalent to each calorie multiplied by one thousand. A kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with the “Calorie” (with a capital

In [12]:
#define splits size to chunks groups of sentences to chunks of sentence
splits_sentence_chunks = 10

def splits_sentence_to_chunks(input_list: list[str],chunk_size: int):
  return [input_list[i:i+chunk_size] for i in range(0,len(input_list),chunk_size)]  #here i =0,10,20 increases by multiple of 10 in each loop


for items in pages_and_text:
  items["sentence_chunks"] = splits_sentence_to_chunks(items["sentence"],splits_sentence_chunks)
  items["sentence_chunks_count"] = len(items["sentence_chunks"])

pages_and_text[20:24]


[{'page_number': 1,
  'page_character_count': 66,
  'page_word_count': 12,
  'page_sentence_count': 1,
  'text': 'PART I BASIC CONCEPTS IN NUTRITION BASIC CONCEPTS IN NUTRITION | 1',
  'sentence': ['PART I BASIC CONCEPTS IN NUTRITION BASIC CONCEPTS IN NUTRITION | 1'],
  'sentence_count': 1,
  'sentence_chunks': [['PART I BASIC CONCEPTS IN NUTRITION BASIC CONCEPTS IN NUTRITION | 1']],
  'sentence_chunks_count': 1},
 {'page_number': 2,
  'page_character_count': 31,
  'page_word_count': 6,
  'page_sentence_count': 1,
  'text': '2 | BASIC CONCEPTS IN NUTRITION',
  'sentence': ['2 | BASIC CONCEPTS IN NUTRITION'],
  'sentence_count': 1,
  'sentence_chunks': [['2 | BASIC CONCEPTS IN NUTRITION']],
  'sentence_chunks_count': 1},
 {'page_number': 3,
  'page_character_count': 353,
  'page_word_count': 60,
  'page_sentence_count': 4,
  'text': 'INTRODUCTION “If you’re hungry, you better eat what you know. If you’re from New Orleans and you get really hungry, you better have New Orleans food.” – Ch

# Splitting each chunks into its own items.


Lets create a pages_and_chunks list where each sentence chunks are divided into pages_and_chunks and count its items respectivley.

In [13]:
import re

pages_and_chunks = []

for items in pages_and_text:
  for sentence in items["sentence_chunks"]:
    chunks_dict = {}
    chunks_dict["page_number"] = items["page_number"]
    joined_sentence = "".join(sentence).strip()
    joined_sentence = re.sub(r'\.([A-Z])',r'. \1',joined_sentence)


    chunks_dict["sentence_chunk"] = joined_sentence

    chunks_dict["chunk_char_count"] = len(joined_sentence)
    chunks_dict["chunk_word_count"] = len(joined_sentence.split())
    chunks_dict["chunk_token_count"] = len(joined_sentence)/4
    pages_and_chunks.append(chunks_dict)

In [14]:
pages_and_chunks[20:25]

[{'page_number': -1,
  'sentence_chunk': 'This page provides a record of changes made to this textbook. New editions are released when significant edits are made to the textbook. Revised editions are acknowledged with an e after the edition number (ex.2e). The exported files for this textbook reflect the most recent version. Table i. Human Nutrition OER Textbook Version History Version Date Details 1 June 4, 2018 Human Nutrition textbook published by the University of Hawai‘i at Mānoa’s Outreach College.2 August 27, 2020 Second edition of the Human Nutrition textbook published by the University of Hawai‘i at Mānoa’s Outreach College. •Added Health at Every Size section (Chapter 18) • Nutrition Facts Label section updated to reflect new guidelines (Chapter 12) • Major and Trace Minerals, Pregnancy and Infancy sections expanded (Chapters 10, 11, 13) • Embedded 176 learning activities created with the open source tool H5P ◦ Over 100 flashcards were added at the end of relevant sections ◦ 

In [15]:
len(pages_and_chunks)

1752

In [16]:
import pandas as pd

df = pd.DataFrame(pages_and_chunks)
df.head(24)

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,-19,Human Nutrition,15,2,3.75
1,-17,HUMAN NUTRITION Louisiana Edition MELISSA JOHN...,105,14,26.25
2,-16,Human Nutrition Copyright © 2022 by Melissa Jo...,238,30,59.50
3,-15,CONTENTS Preface xiii About the Contributors x...,518,79,129.50
4,-14,The Endocrine System 85 The Urinary System 89 ...,767,123,191.75
5,-13,Part V. Protein Introduction 219 Defining Prot...,646,93,161.50
6,-12,Part VIII. Water and Electrolytes Introduction...,559,80,139.75
7,-11,Magnesium 493 Summary of Major Minerals 497 Pa...,442,67,110.50
8,-10,Adolescence 648 Late Adolescence 652 Part XIV....,656,92,164.00
9,-9,Part XVII. Food Safety Introduction 773 The Ma...,636,97,159.00


In [17]:
min_chunk = min(chunk["chunk_token_count"] for chunk in pages_and_chunks)
max_chunk = max(chunk["chunk_token_count"] for chunk in pages_and_chunks)
print("min",min_chunk)
print("max",max_chunk)

min 2.5
max 653.0


In [18]:
import random
random.sample(pages_and_chunks,k=1)

[{'page_number': 24,
  'sentence_chunk': 'Vegan. This vegetarian diet does not include dairy, eggs, or any animal products or by-products. Practicing the Principles As Louisiana is known for its seafood and other meats, adopting a vegetarian or vegan diet may be an intimidating feat. However, it is not impossible!It’s time to put your chef’s hat on and 24 | LIFESTYLES AND NUTRITION',
  'chunk_char_count': 342,
  'chunk_word_count': 57,
  'chunk_token_count': 85.5}]

In [19]:
#Lets see the data(token chunks) that are above than 2000 and lower than 50

number_of_chunks = 0
for chunk in pages_and_chunks:
  if chunk["chunk_token_count"] > 500:
    number_of_chunks += 1
    print(chunk)
print(number_of_chunks)


{'page_number': 305, 'sentence_chunk': 'organizations were actively reinforcing strategies aimed at meeting the challenge of improving the health of all Americans. In 2010, the national campaign to reduce obesity was reinforced when First Lady Michelle Obama launched the “Let’s Move” initiative, which has the goal of “solving the challenge of childhood obesity within a generation so that children born today will reach adulthood at a healthy weight.”8 Another campaign, “Campaign to End Obesity,” was recently established to try to enable more Americans to eat healthy and be active by bringing together leaders from academia and industry, as well as public-health policy-makers, to create policies that will reverse the obesity trend and its associated diseases. The “Small-Change” Approach Currently, most people are not obese in this country. The gradual rise in overweight is happening because, on average, people consume slightly more calories daily than they expend, resulting in a gradual w

Now filter the token less than 30 and more than 500.Since less than 30 are more like headings,intro and references.And more than 500 too large and topics mixed.

In [20]:
filtered_chunks = []

for chunk in pages_and_chunks:
  if 30<= chunk["chunk_token_count"] <= 500:
    filtered_chunks.append(chunk)

print(len(pages_and_chunks))
print(len(filtered_chunks))


1752
1571


In [21]:
#Now lets see the chunks
import random
random.sample(filtered_chunks,k=1)

[{'page_number': 147,
  'sentence_chunk': 'fiber meal as compared to a low-fiber meal (see Figure 4.14 “Fibers Role in Carbohydrate Digestion and Absorption”) can significantly slow down the absorption process, therefore affecting blood glucose levels. Americans typically do not consume the recommended amount of whole grains, which is 50 percent or more of grains from whole grains. Figure 3.14 Fibers Role in Carbohydrate Digestion and Absorption (Source: University of Hawaii @ Manoa [New Tab], Allison Calabrese, CC-BY-NC-SA) Diets high in whole grains have repeatedly been shown to decrease weight. A large group of studies all support that consuming more than two servings of whole grains per day reduces one’s chances of getting Type 2 diabetes by 21 percent.7 The Nurses’ Health Study found that women who consumed two to three servings of whole grain products daily were 30 percent less likely to have a heart attack.8 7.de Munter, J. S., Hu, F. B., et al. (2007). Whole Grain, Bran, and Ge

Now,Lets Embeddings the filterd chunks we prepare earlier.So, to embeds use the all-mpnet-base-v2 from the hugging face transformer.

In [22]:
!pip install sentence-transformers

In [23]:
from sentence_transformers import SentenceTransformer

#load pre-trained embedding model
model = SentenceTransformer("all-mpnet-base-v2")

#prepare sentence or chunks take one example
sentences = [
    "Human Nutrition textbook published by University of Hawai‘i at Mānoa.",
    "This chapter explains vitamins and minerals.",
    "Seafood gumbo is made with roux, shrimp, and vegetables."
]

#Encode sentence into embeddings
embeddings = model.encode(sentence)
print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(8, 768)


In [24]:
print(embeddings)

[[ 0.08436868  0.02907833  0.07351795 ... -0.02596605 -0.00244565
  -0.01972943]
 [-0.01307861  0.0561556  -0.0083619  ... -0.03817027  0.04548811
  -0.01025813]
 [ 0.05974299 -0.12097659  0.00187955 ...  0.01324266  0.01763458
  -0.00459644]
 ...
 [ 0.07914154 -0.02192187 -0.0376835  ... -0.03706811 -0.05492734
  -0.07540243]
 [ 0.02323637  0.01840339 -0.02880682 ... -0.03250247  0.01334669
  -0.06288543]
 [ 0.05605077 -0.01007421 -0.02405846 ... -0.01976281 -0.05055379
  -0.03034719]]


Lets use embeddings into our filtered chunks

In [25]:

model.to("cuda")

#Embed each chunk one by one
for chunk in filtered_chunks:
  chunk["chunk_embedding"] = model.encode(chunk["sentence_chunk"])

In [26]:
filtered_chunks[20]

{'page_number': 4,
 'sentence_chunk': 'These basic functions allow us to detect and respond to our environmental surroundings, move, excrete wastes, respire (breathe), grow, and reproduce. Six nutrients are required for the body to function and maintain overall health (Figure 1.1). These are carbohydrates, lipids, proteins, water, vitamins, and minerals. Foods also contain non-nutrients that may be harmful (such as natural toxins common in plant foods and additives like some dyes and preservatives) or beneficial (such as antioxidants). Figure 1.1 The six major nutrient classes Macronutrients Micronutrients 4 | INTRODUCTION',
 'chunk_char_count': 591,
 'chunk_word_count': 85,
 'chunk_token_count': 147.75,
 'chunk_embedding': array([ 4.54076454e-02,  1.89694623e-03,  7.00056646e-03,  1.68529134e-02,
         5.35846166e-02,  2.83740461e-02, -4.59137633e-02,  1.89810954e-02,
         4.91336882e-02, -6.71576485e-02, -5.17519191e-03,  1.17862510e-04,
         1.48754893e-02,  1.01096407e-0

In [27]:
text_chunks = [chunks["sentence_chunk"] for chunks in filtered_chunks]

text_chunks[20]

'These basic functions allow us to detect and respond to our environmental surroundings, move, excrete wastes, respire (breathe), grow, and reproduce. Six nutrients are required for the body to function and maintain overall health (Figure 1.1). These are carbohydrates, lipids, proteins, water, vitamins, and minerals. Foods also contain non-nutrients that may be harmful (such as natural toxins common in plant foods and additives like some dyes and preservatives) or beneficial (such as antioxidants). Figure 1.1 The six major nutrient classes Macronutrients Micronutrients 4 | INTRODUCTION'

In [28]:
#only embeddings of text
text_chunks_embeddings = model.encode(
    text_chunks,
    batch_size = 32,
    convert_to_tensor= True
)

In [29]:
text_chunks_embeddings.shape

torch.Size([1571, 768])

In [30]:
text_chunks_embeddings[20]

tensor([ 4.5408e-02,  1.8969e-03,  7.0005e-03,  1.6853e-02,  5.3585e-02,
         2.8374e-02, -4.5914e-02,  1.8981e-02,  4.9134e-02, -6.7158e-02,
        -5.1751e-03,  1.1791e-04,  1.4876e-02,  1.0110e-02,  2.4925e-02,
        -3.2867e-02,  2.8692e-02, -2.6547e-03, -5.9718e-02,  8.1612e-03,
        -4.4347e-02,  2.0827e-02,  3.2999e-03, -3.0291e-02, -5.0766e-02,
         6.4247e-02,  3.0010e-02,  1.7510e-02,  7.1003e-03, -1.3848e-02,
        -3.9092e-02,  9.6753e-03,  5.0459e-03, -8.5709e-02,  2.3791e-06,
        -1.5444e-04, -5.0604e-02, -7.1479e-03, -4.4965e-02, -1.3202e-02,
         1.8865e-02, -2.3606e-02, -4.7650e-03, -2.0552e-03,  5.1250e-02,
         9.2395e-03,  1.0010e-02,  5.8825e-02,  1.3940e-02,  2.0032e-02,
         1.5656e-02,  2.6093e-02,  3.2896e-03,  5.6863e-03, -3.9753e-02,
         6.2479e-02,  2.5482e-02,  7.5823e-02,  2.0403e-02,  4.8822e-03,
         1.1350e-02,  3.1996e-02,  2.4207e-02,  3.9209e-02,  1.7272e-02,
         6.8537e-02, -1.8003e-02, -3.9619e-02,  4.5

# Save text and embeddings to csv file.

In [31]:
text_chunks_and_embeddings_df = pd.DataFrame(filtered_chunks)
text_chunks_and_embeddings_df.to_csv("text_chunks_and_embeddings_df.csv",index=False)

In [32]:
#Import the csv file and see data
text_chunk_and_embeddings_load = pd.read_csv("text_chunks_and_embeddings_df.csv")
text_chunk_and_embeddings_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,chunk_embedding
0,-16,Human Nutrition Copyright © 2022 by Melissa Jo...,238,30,59.50,[ 3.88624743e-02 5.59770241e-02 -5.76988840e-...
1,-15,CONTENTS Preface xiii About the Contributors x...,518,79,129.50,[ 4.28345352e-02 4.43807244e-02 5.18328277e-...
2,-14,The Endocrine System 85 The Urinary System 89 ...,767,123,191.75,[ 5.54008633e-02 4.40658145e-02 1.09777590e-...
3,-13,Part V. Protein Introduction 219 Defining Prot...,646,93,161.50,[ 5.20048738e-02 1.00022936e-02 5.58838341e-...
4,-12,Part VIII. Water and Electrolytes Introduction...,559,80,139.75,[ 4.85534072e-02 -1.86877344e-02 -3.01864967e-...


# Similarity Search

-Similarity search or semantic search or vector search is the idea of searching on *vibe*.

If this sounds like woo, woo. It's not.

Perhaps searching via *meaning* is a better analogy.

With keyword search, you are trying to match the string "apple" with the string "apple".

Whereas with similarity/semantic search, you may want to search "macronutrients functions".

And get back results that don't necessarily contain the words "macronutrients functions" but get back pieces of text that match that meaning.

> **Example:** Using similarity search on our textbook data with the query "macronutrients function" returns a paragraph that starts with:
>
>*There are three classes of macronutrients: carbohydrates, lipids, and proteins. These can be metabolically processed into cellular energy. The energy from macronutrients comes from their chemical bonds. This chemical energy is converted into cellular energy that is then utilized to perform work, allowing our bodies to conduct their basic functions.*

In [33]:
import random

import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

#import text and embedding df
text_chunks_and_embeddings_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

#convert embeddings column back to np array(it got converted to string when it got saved to csv)
text_chunks_and_embeddings_df["chunk_embedding"] = text_chunks_and_embeddings_df["chunk_embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

#convert embeddings to torch tensor and send to the device(note: Numpy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embeddings_df["chunk_embedding"].tolist()), dtype=torch.float32).to(device)

embeddings.shape

torch.Size([1571, 768])

now we have small data like 1571 chunking so we donot need to batch.Also we only search similarity not train model.So donot need batching.

For RAG simple pipeline<br>
1.Define a query<br>
2.Embeddings of query(Note: must use same embedding model to embed the query which we embed the sentence chunk)<br>
3.Perform a dot product or cosine similarity function over a query embeddings and text embeddings.<br>
4.Sort the result from desending order using top- k.

In [99]:
#define a query
import torch
import torch.nn.functional as F

query = "What are Macronutrients and What are thier roles?"
print(f"query:{query}")

#Embeddings of query
query_embedings = model.encode(query,convert_to_tensor=True).to(device)

#dot product of query and text embeddings
similarity = F.cosine_similarity(embeddings,query_embedings)

#Get the top k results
top_result_similarity = torch.topk(similarity,k=3)

top_result_similarity


query:What are Macronutrients and What are thier roles?


torch.return_types.topk(
values=tensor([0.7079, 0.6692, 0.6572], device='cuda:0'),
indices=tensor([21, 27, 26], device='cuda:0'))

In [100]:
filtered_chunks[21]

{'page_number': 5,
 'sentence_chunk': 'Macronutrients Nutrients that are needed in large amounts are called macronutrients. There are three classes of macronutrients: carbohydrates, lipids, and proteins. Through various biological processes occurring in the body, these nutrients can be metabolized to produce cellular energy. The energy from macronutrients comes from their chemical bonds. This chemical energy is converted into cellular energy that is then utilized to perform work, allowing our bodies to conduct their basic functions. The energy obtained from food, which thus becomes cellular energy, is measured in calorie units. A unit of measurement of food energy is the calorie. On nutrition food labels, the amount given for “calories” is actually equivalent to each calorie multiplied by one thousand. A kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with the “Calorie” (with a capital “C”) on nutrition food labels. Although water does not yield calories, it

Use vector db and FAISS and ANN(Nearest Neighbor) for the big data more than 1M to calculate similarity.

Now make our vector ouput more pretty.

In [101]:
import textwrap
def print_wrapped(text,wrap_length=80):
  wrapped_text = textwrap.fill(text,wrap_length)
  print(wrapped_text)

# this func do Text automatically broken into multiple lines
#Each line ≤ 50 characters
#Makes text readable in console / terminal

In [102]:
query = "What are Macronutrients and what roles they play?"
print(f"query:{query}\n")
print("Result:")

#Loop through zip together scores and indices from torch.topk
#top k garda indices,values ma aauxa so hamle top_result_similarity[0] and 1 use gareko for values and indices.
for score,idx in zip(top_result_similarity[0],top_result_similarity[1]):
  print(f"Score:{score:.4f}")
  print("Text:")
  print_wrapped(filtered_chunks[idx]["sentence_chunk"])
  print(f"Page Number:{filtered_chunks[idx]["page_number"]}")
  print("\n")


query:What are Macronutrients and what roles they play?

Result:
Score:0.7079
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. Through various biological processes occurring in the
body, these nutrients can be metabolized to produce cellular energy. The energy
from macronutrients comes from their chemical bonds. This chemical energy is
converted into cellular energy that is then utilized to perform work, allowing
our bodies to conduct their basic functions. The energy obtained from food,
which thus becomes cellular energy, is measured in calorie units. A unit of
measurement of food energy is the calorie. On nutrition food labels, the amount
given for “calories” is actually equivalent to each calorie multiplied by one
thousand. A kilocalorie (one thousand calories, denoted with a small “c”) is
synonymous with the “Calorie” (with a capital “C”) on nutrition food label

Now lets use the reranking process.Means feri scores predict garxa ani highest scores lai first and desending.Predict garda query + chunks duitai ko score predict hunxa.

In [38]:
from sentence_transformers import CrossEncoder

#Load rerank model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")




config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [39]:
query = "How to reduce the Risk of Cardiovascular Disease?"
# Extract top chunks (full dictionaries) using integer indices from filtered_chunks
top_chunks_dicts = [filtered_chunks[i.item()] for i in top_result_similarity[1]]

# Now pair query with the 'sentence_chunk' text from the extracted dictionaries
pairs = [[query, chunk_dict["sentence_chunk"]] for chunk_dict in top_chunks_dicts]

#print(pairs)

# Rerank scores
rerank_scores = reranker.predict(pairs)

# Now sort reranked scores along with the original chunk dictionaries
sorted_rerank_scores = sorted(zip(rerank_scores, top_chunks_dicts),
                                  key=lambda x: x[0],
                                  reverse=True)

for score, chunk_dict in sorted_rerank_scores:
  print(f"Score:{score:.4f}")
  print("Text:")
  print_wrapped(chunk_dict["sentence_chunk"])
  print(f"Page Number:{chunk_dict["page_number"]}")
  print("\n")

Score:-10.4124
Text:
Learning Objectives By the end of this chapter you will be able to: • Describe
the physiological changes that occur in response to exercise • Describe the
effects of physical fitness on overall health • Describe the purpose and
applications of nutrition supplements Becoming and staying physically fit is an
important part of achieving optimal health. A well-rounded exercise program is
crucial to becoming and remaining healthy. Physical activity improves your
health in a number of ways. It promotes weight loss, strengthens muscles and
bones, keeps the heart and lungs strong, and helps to protect against chronic
disease. There are four essential elements of physical fitness: cardio-
respiratory, muscular strength, flexibility, and maintaining a healthful body
composition. Some enthusiasts might argue the relative importance of each, but
optimal health requires some degree of balance between all four. For example
Louisiana offers a variety of endurance events throughou

Lets make all this in function like return top k value and indices in func.

In [103]:
def retrieve_relavent_resources(query: str,embeddings: torch.tensor,model:SentenceTransformer= model):
  query_embeddings = model.encode(query,convert_to_tensor=True)

  #Get cosine similarity
  similarity = F.cosine_similarity(embeddings,query_embedings)

  #Get the top k results
  scores,indices = torch.topk(similarity,k=5)

  return scores,indices

In [104]:
def top_results_and_scores(query: str,embeddings: torch.tensor,filtered_chunks: list[dict]=filtered_chunks):
  scores,indices = retrieve_relavent_resources(query=query,embeddings=embeddings)
  for score,idx in zip(scores,indices):
    print(f"Score:{score:.4f}")
    print("Text:")
    print_wrapped(filtered_chunks[idx]["sentence_chunk"])
    print(f"Page Number:{filtered_chunks[idx]["page_number"]}")
    print("\n")


In [105]:
for item in filtered_chunks:
  if item["page_number"]==22:
    print(item["sentence_chunk"])



advisory that urged a warning be included on alcoholic beverages to state the potential risk of cancer associated with alcoholic beverage consumption.6 Illicit and prescription drug abuse are associated with decreased health and are a prominent problem in the United States. The health effects of drug abuse can be far-reaching, including the increased risk of stroke, heart disease, cancer, lung disease, and liver disease.7 Sleeping Patterns Inadequate amounts of sleep, or not sleeping well, can also have remarkable effects on a person’s health. In fact, sleeping can affect your health just as much as your diet. Scientific studies have shown that insufficient sleep increases the risk for heart disease, Type 2 diabetes, obesity, and depression. Abnormal breathing during sleep, a condition called sleep apnea, is also linked to an increased risk for chronic disease.8 Personal Choice: The Challenge of Choosing Foods Other factors besides environment and lifestyle influence the foods you choo

In [106]:
query = "What are macronutrients and what are their roles?"
#retrieve_relavent_resources(query=query,embeddings=embeddings)
top_results_and_scores(query=query,embeddings=embeddings,filtered_chunks=filtered_chunks)

Score:0.7079
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. Through various biological processes occurring in the
body, these nutrients can be metabolized to produce cellular energy. The energy
from macronutrients comes from their chemical bonds. This chemical energy is
converted into cellular energy that is then utilized to perform work, allowing
our bodies to conduct their basic functions. The energy obtained from food,
which thus becomes cellular energy, is measured in calorie units. A unit of
measurement of food energy is the calorie. On nutrition food labels, the amount
given for “calories” is actually equivalent to each calorie multiplied by one
thousand. A kilocalorie (one thousand calories, denoted with a small “c”) is
synonymous with the “Calorie” (with a capital “C”) on nutrition food labels.
Although water does not yield calories, it is considered to be

Getting an LLM for local generation

We're got our retrieval pipeline ready, let's now get the generation side of things happening.

To perform generation, we're going to use a Large Language Model (LLM).

LLMs are designed to generate an output given an input.

In our case, we want our LLM to generate and output of text given a input of text.

And more specifically, we want the output of text to be generated based on the context of relevant information to the query.

The input to an LLM is often referred to as a prompt.

We'll augment our prompt with a query as well as context from our textbook related to that query.

> **Which LLM should I use?**

There are many LLMs available.

Two of the main questions to ask from this is:
1. Do I want it to run locally?
2. If yes, how much compute power can I dedicate?

If you're after the absolute best performance, you'll likely want to use an API (not running locally) such as GPT-4 or Claude 3. However, this comes with the tradeoff of sending your data away and then awaiting a response.

For our case, since we want to set up a local pipeline and run it on our own GPU, we'd answer "yes" to the first question and then the second question will depend on what hardware we have available.

To find open-source LLMs, one great resource is the [Hugging Face open LLM leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard).

The leaderboard compares many of the latest and greatest LLMs on various benchmarks.

Another great resource is [TheBloke on Hugging Face](https://huggingface.co/TheBloke), an account which provides an extensive range of quantized (models that have been made smaller) LLMs.

A rule of thumb for LLMs (and deep learning models in general) is that the higher the number of parameters, the better the model performs.

It may be tempting to go for the largest size model (e.g. a 70B parameter model rather than a 7B parameter model) but a larger size model may not be able to run on your available hardware.

The following table gives an insight into how much GPU memory you'll need to load an LLM with different sizes and different levels of [numerical precision](https://en.wikipedia.org/wiki/Precision_(computer_science)).

They are based on the fact that 1 float32 value (e.g. `0.69420`) requires 4 bytes of memory and 1GB is approximately 1,000,000,000 (one billion) bytes.

| Model Size (Billion Parameters) | Float32 VRAM (GB) | Float16 VRAM (GB) | 8-bit VRAM (GB) | 4-bit VRAM (GB) |
|-----|-----|-----|-----|-----|
| 1B                              | ~4                | ~2                | ~1              | ~0.5            |
| 7B (e.g., [Llama 2 7B](https://huggingface.co/meta-llama/Llama-2-7b), [Gemma 7B](https://huggingface.co/google/gemma-7b-it), [Mistral 7B](https://huggingface.co/mistralai/Mistral-7B-v0.1))             | ~28               | ~14               | ~7              | ~3.5            |
| 10B                             | ~40               | ~20               | ~10             | ~5              |
| 70B (e.g, Llama 2 70B)          | ~280              | ~140              | ~70             | ~35             |
| 100B                            | ~400              | ~200              | ~100            | ~50             |
| 175B                            | ~700              | ~350              | ~175            | ~87.5           |

<br>

> **Note:** Loading a model in a lower precision (e.g. 8-bit instead of float16) generally lowers performance. Lower precision can help to reduce computing requirements, however sometimes the performance degradation in terms of model output can be substantial. Finding the right speed/performance tradeoff will often require many experiments.

### Checking local GPU memory availability

Let's find out what hardware we've got available and see what kind of model(s) we'll be able to load.

> **Note:** You can also check this with the `!nvidia-smi` command.

In [44]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 15 GB


Ok wonderful!

I'm running this notebook with a NVIDIA RTX 4090, so I've got 15GB of VRAM available.

However, this may be different on your end.

Looking at the table above, it seems we can run a ~7-10B parameter model in float4 precision pretty comfortably.

But we could also run a smaller one if we'd like.

Let's try out the recently released (at the time of writing, March 2024) LLM from Google, [Gemma](https://huggingface.co/blog/gemma).

Specifically, we'll use the `gemma-7b-it` version which stands for Gemma 7B Instruction-Tuned.

Instruction tuning is the process of tuning a raw language model to follow instructions.

These are the kind of models you'll find in most chat-based assistants such as ChatGPT, Gemini or Claude.

The following table shows different amounts of GPU memory requirements for different verions of the Gemma LLMs with varying levels of precision.

| Model             | Precision | Min-Memory (Bytes) | Min-Memory (MB) | Min-Memory (GB) | Recommended Memory (GB) | Hugging Face ID |
|-------------------|-----------|----------------|-------------|-------------| ----- | ----- |
| [Gemma 2B](https://huggingface.co/google/gemma-2b-it)          | 4-bit     | 2,106,749,952  | 2009.15     | 1.96        | ~5.0 | [`gemma-2b`](https://huggingface.co/google/gemma-2b) or [`gemma-2b-it`](https://huggingface.co/google/gemma-2b-it) for instruction tuned version |
| Gemma 2B          | Float16   | 5,079,453,696  | 4844.14     | 4.73        | ~8.0 | Same as above |
| [Gemma 7B](https://huggingface.co/google/gemma-7b-it)          | 4-bit     | 5,515,859,968  | 5260.33     | 5.14        | ~8.0 | [`gemma-7b`](https://huggingface.co/google/gemma-7b) or [`gemma-7b-it`](https://huggingface.co/google/gemma-7b-it) for instruction tuned version |
| Gemma 7B          | Float16   | 17,142,470,656 | 16348.33    | 15.97       | ~19 | Same as above |

> **Note:** `gemma-7b-it` means "instruction tuned", as in, a base LLM (`gemma-7b`) has been fine-tuned to follow instructions, similar to [`Mistral-7B-v0.1`](https://huggingface.co/mistralai/Mistral-7B-v0.1) and [`Mistral-7B-Instruct-v0.1`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1).
>
> There are also further quantized and smaller variants of Gemma (and other LLMs) available in various formats such as GGUF. You can see many of these on [TheBloke account on Hugging Face](https://huggingface.co/TheBloke).
>
> The version of LLM you choose to use will be largely based on project requirements and experimentation.

Based on the table above, let's write a simple if/else statement which recommends which Gemma variant we should look into using.

In [45]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
    print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
    use_quantization_config = True
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
    use_quantization_config = False
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
    use_quantization_config = False
    model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

GPU memory: 15 | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.
use_quantization_config set to: False
model_id set to: google/gemma-2b-it


### Loading an LLM locally

Alright! Looks like `gemma-7b-it` it is (for my local machine with an RTX 4090, change the `model_id` and `use_quantization_config` values to suit your needs)!

There are plenty of examples of how to load the model on the `gemma-7b-it` [Hugging Face model card](https://huggingface.co/google/gemma-7b-it).

Good news is, the Hugging Face [`transformers`](https://huggingface.co/docs/transformers/) library has all the tools we need.

To load our LLM, we're going to need a few things:
1. A quantization config (optional) - This will determine whether or not we load the model in 4bit precision for lower memory usage. The we can create this with the [`transformers.BitsAndBytesConfig`](https://huggingface.co/docs/transformers/v4.38.2/en/main_classes/quantization#transformers.BitsAndBytesConfig) class (requires installing the [`bitsandbytes` library](https://github.com/TimDettmers/bitsandbytes)).
2. A model ID - This is the reference Hugging Face model ID which will determine which tokenizer and model gets used. For example `gemma-7b-it`.
3. A tokenzier - This is what will turn our raw text into tokens ready for the model. We can create it using the [`transformers.AutoTokenzier.from_pretrained`](https://huggingface.co/docs/transformers/v4.38.2/en/model_doc/auto#transformers.AutoTokenizer) method and passing it our model ID.
4. An LLM model - Again, using our model ID we can load a specific LLM model. To do so we can use the [`transformers.AutoModelForCausalLM.from_pretrained`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForCausalLM.from_pretrained) method and passing it our model ID as well as other various parameters.

As a bonus, we'll check if [Flash Attention 2](https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2) is available using `transformers.utils.is_flash_attn_2_available()`. Flash Attention 2 speeds up the attention mechanism in Transformer architecture models (which is what many modern LLMs are based on, including Gemma). So if it's available and the model is supported (not all models support Flash Attention 2), we'll use it. If it's not available, you can install it by following the instructions on the [GitHub repo](https://github.com/Dao-AILab/flash-attention).

> **Note:** Flash Attention 2 currently works on NVIDIA GPUs with a compute capability score of 8.0+ (Ampere, Ada Lovelace, Hopper architectures). We can check our GPU compute capability score with [`torch.cuda.get_device_capability(0)`](https://pytorch.org/docs/stable/generated/torch.cuda.get_device_capability.html).

> **Note:** To get access to the Gemma models, you will have to [agree to the terms & conditions](https://huggingface.co/google/gemma-7b-it) on the Gemma model page on Hugging Face. You will then have to authorize your local machine via the [Hugging Face CLI/Hugging Face Hub `login()` function](https://huggingface.co/docs/huggingface_hub/en/quick-start#authentication). Once you've done this, you'll be able to download the models. If you're using Google Colab, you can add a [Hugging Face token](https://huggingface.co/docs/hub/en/security-tokens) to the "Secrets" tab.
>
> Downloading an LLM locally can take a fair bit of time depending on your internet connection. Gemma 7B is about a 16GB download and Gemma 2B is about a 6GB download.

Let's do it!

In [107]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available

# 1. Create quantization config for smaller model loading (optional)
# Requires !pip install bitsandbytes accelerate, see: https://github.com/TimDettmers/bitsandbytes, https://huggingface.co/docs/accelerate/
# For models that require 4-bit quantization (use this if you have low GPU memory available)
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16)

# Bonus: Setup Flash Attention 2 for faster inference, default to "sdpa" or "scaled dot product attention" if it's not available
# Flash Attention 2 requires NVIDIA GPU compute capability of 8.0 or above, see: https://developer.nvidia.com/cuda-gpus
# Requires !pip install flash-attn, see: https://github.com/Dao-AILab/flash-attention
if (is_flash_attn_2_available()) and (torch.cuda.get_device_capability(0)[0] >= 8):
  attn_implementation = "flash_attention_2"
else:
  attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")

# 2. Pick a model we'd like to use (this will depend on how much GPU memory you have available)
#model_id = "google/gemma-7b-it"
model_id = model_id # (we already set this above)
print(f"[INFO] Using model_id: {model_id}")

# 3. Instantiate tokenizer (tokenizer turns text into numbers ready for the model)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

# 4. Instantiate the model
llm_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id,
                                                 torch_dtype=torch.float16, # datatype to use, we want float16
                                                 quantization_config=quantization_config if use_quantization_config else None,
                                                 low_cpu_mem_usage=False, # use full memory
                                                 attn_implementation=attn_implementation) # which attention version to use

if not use_quantization_config: # quantization takes care of device setting automatically, so if it's not used, send model to GPU
    llm_model.to("cuda")

[INFO] Using attention implementation: sdpa
[INFO] Using model_id: google/gemma-2b-it


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [48]:
llm_model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): GemmaRMSNorm((2048,), 

In [49]:
#test
input_text = "Write a poem about machine learning."
input_ids = tokenizer(input_text,return_tensors="pt").to("cuda")
outputs = llm_model.generate(**input_ids)
print(tokenizer.decode(outputs[0]))

<bos>Write a poem about machine learning.

Machines, they weave and they learn,
From the data, they discern.
Algorithms,


In [50]:
#Nice way to do
input_text = "What are Macronutrients and their functions?"
print(f"Input text:\n{input_text}")

#create prompt template for instruction tuned model
dialogue_template = [
    {
        "role": "user",
        "content": input_text
    }
]

#Apply the chat template
prompt = tokenizer.apply_chat_template(conversation=dialogue_template,tokenize = False,add_generation_prompt=True)

print(f"Prompt Format:\n{prompt}")

Input text:
What are Macronutrients and their functions?
Prompt Format:
<bos><start_of_turn>user
What are Macronutrients and their functions?<end_of_turn>
<start_of_turn>model



In [51]:
#tokenize the input text
input_ids = tokenizer(prompt,return_tensors="pt").to("cuda")
input_ids

{'input_ids': tensor([[     2,      2,    106,   1645,    108,   1841,    708,  97586, 184592,
            578,   1024,   7257, 235336,    107,    108,    106,   2516,    108]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [52]:
#now generate the output
output = llm_model.generate(**input_ids,max_new_tokens = 250)
print(f"Output of model:{output[0]}")

Output of model:tensor([     2,      2,    106,   1645,    108,   1841,    708,  97586, 184592,
           578,   1024,   7257, 235336,    107,    108,    106,   2516,    108,
         21404, 235269,   1517, 235303, 235256,    476,  13367,    576,  97586,
        184592,    578,   1024,   7257, 235292,    109,    688,  12298,   1695,
        184592,    688,    708,  37132,    674,    573,   2971,   4026,    575,
          2910,  15992,    577,  10528,   1426,   2962, 235265,   2365,    708,
         13538,   1280,   1378,   1872,  14486, 235292, 186809, 184592,    578,
         92800, 184592, 235265,    109,    688,  12298,   1695, 184592,    688,
           708,  37132,    674,    573,   2971,   4026,    575,   8107,  15992,
        235265,   2365,    708,  13538,   1280,   2149,  14486, 235292,    109,
        235287,   5231, 156615,  56227,    688,   3658,   4134,    604,    573,
          2971, 235265,   2365,    708,   1942,    575,  16512,   1582,    685,
         11843, 235269, 

In [53]:
#now decode the token into text
output_decoded = tokenizer.decode(output[0])
print(f"Model Output:\n{output_decoded}\n")

Model Output:
<bos><bos><start_of_turn>user
What are Macronutrients and their functions?<end_of_turn>
<start_of_turn>model
Sure, here's a summary of Macronutrients and their functions:

**Macronutrients** are nutrients that the body needs in large amounts to maintain good health. They are divided into two main categories: macronutrients and micronutrients.

**Macronutrients** are nutrients that the body needs in larger amounts. They are divided into three categories:

* **Carbohydrates** provide energy for the body. They are found in foods such as bread, pasta, rice, and potatoes.
* **Proteins** are used to build and repair tissues, produce enzymes, and make hormones. They are found in foods such as meat, fish, eggs, and beans.
* **Fats** provide energy, help absorb vitamins, and insulate the body. They are found in foods such as oil, butter, nuts, and seeds.

**Micronutrients** are nutrients that the body needs in smaller amounts. They are divided into two categories:

* **Minerals** 

Lets make a list of query we can try out for our pipelines.

In [108]:
# Nutrition-style questions generated with GPT4
gpt5_questions = [
    "What are the macronutrients, and what roles do they play in the human body?",
    "What is Osteoporosis, and what causes it?",
    "Describe the process of digestion and absorption of nutrients in the human body.",
    "How to reduce the Risk of Cardiovascular Disease?",
    "Explain the concept of energy balance and its importance in weight management.?"
]

# Manually created question list
manual_questions = [
     "What are the main classes of macronutrients?",
      "What are the steps to reducing the risk of cancer?",
      "What are the types of diabetes?",
      "What is Gestational Diabetes?",
      "How much water should an adult consume daily?"
]

In [109]:
query_list = gpt5_questions + manual_questions
query_list

['What are the macronutrients, and what roles do they play in the human body?',
 'What is Osteoporosis, and what causes it?',
 'Describe the process of digestion and absorption of nutrients in the human body.',
 'How to reduce the Risk of Cardiovascular Disease?',
 'Explain the concept of energy balance and its importance in weight management.?',
 'What are the main classes of macronutrients?',
 'What are the steps to reducing the risk of cancer?',
 'What are the types of diabetes?',
 'What is Gestational Diabetes?',
 'How much water should an adult consume daily?']

In [110]:
#lets generate relevant chunks in context of list of chunks we craeted earlier.
import random
query = random.choice(query_list)
scores,indices = retrieve_relavent_resources(query=query,embeddings=embeddings)
scores,indices

(tensor([0.7079, 0.6692, 0.6572, 0.6458, 0.6432], device='cuda:0'),
 tensor([21, 27, 26, 19, 20], device='cuda:0'))

# Agument Our query to the LLM for the best respose of our Query.

For augmentation we need to join the user query and retrival chunk then pass to LLM.

In [57]:
def prompt_formatter(query: str,retrieve_context: list[dict]):
  context = "- " + "\n-".join([item["sentence_chunk"] for item in retrieve_context])
#   base_prompt = f"""
# Based on the following context items, please answer the query.
# Give yourself room to think by extracting relevant passages from the context before answering the query.
# Don't return the thinking, only return the answer.
# Make sure your answers are as explanatory as possible.
# Use the following examples as reference for the ideal answer style.
# \nExample 1:
# Query: What are the fat-soluble vitamins?
# Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, Vitamin E, and Vitamin K. These vitamins are absorbed along with fats in the diet and can be stored in the body's fatty tissue and liver for later use. Vitamin A is important for vision, immune function, and skin health. Vitamin D plays a critical role in calcium absorption and bone health. Vitamin E acts as an antioxidant, protecting cells from damage. Vitamin K is essential for blood clotting and bone metabolism.
# \nExample 2:
# Query: What are the causes of type 2 diabetes?
# Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity. Factors include a diet high in refined sugars and saturated fats, which can lead to insulin resistance, a condition where the body's cells do not respond effectively to insulin. Over time, the pancreas cannot produce enough insulin to manage blood sugar levels, resulting in type 2 diabetes. Additionally, excessive caloric intake without sufficient physical activity exacerbates the risk by promoting weight gain and fat accumulation, particularly around the abdomen, further contributing to insulin resistance.
# \nExample 3:
# Query: What is the importance of hydration for physical performance?
# Answer: Hydration is crucial for physical performance because water plays key roles in maintaining blood volume, regulating body temperature, and ensuring the transport of nutrients and oxygen to cells. Adequate hydration is essential for optimal muscle function, endurance, and recovery. Dehydration can lead to decreased performance, fatigue, and increased risk of heat-related illnesses, such as heat stroke. Drinking sufficient water before, during, and after exercise helps ensure peak physical performance and recovery.
# \nNow use the following context items to answer the user query:
# {context}
# \nRelevant passages: <extract relevant passages from the context here>
# User query: {query}

# Answer: """

  base_prompt = f"""
  You are a helpful AI assistant.

  Use the following context to answer the question.
  If the answer is not in the context, say "I don't know".

  Context:
  {context}

  Question:
  {query}

  Answer:

  """

  print(f"Query:\n{query}")
  dialogue_template = [
      {
      "role": "user",
      "content":base_prompt
      }
      ]

  prompt = tokenizer.apply_chat_template(conversation=dialogue_template,tokenize=False,add_generation_prompt=True)
  return prompt





In [111]:
query = random.choice(query_list)

# Get relevant resources
scores, indices = retrieve_relavent_resources(query=query,
                                              embeddings=embeddings)

# Create a list of context items
retrieve_context = [filtered_chunks[i] for i in indices]

# Format prompt with context items
prompt = prompt_formatter(query=query,retrieve_context=retrieve_context)
print(prompt)

Query:
What is Osteoporosis, and what causes it?
<bos><start_of_turn>user
You are a helpful AI assistant.

  Use the following context to answer the question.
  If the answer is not in the context, say "I don't know".

  Context:
  - Macronutrients Nutrients that are needed in large amounts are called macronutrients. There are three classes of macronutrients: carbohydrates, lipids, and proteins. Through various biological processes occurring in the body, these nutrients can be metabolized to produce cellular energy. The energy from macronutrients comes from their chemical bonds. This chemical energy is converted into cellular energy that is then utilized to perform work, allowing our bodies to conduct their basic functions. The energy obtained from food, which thus becomes cellular energy, is measured in calorie units. A unit of measurement of food energy is the calorie. On nutrition food labels, the amount given for “calories” is actually equivalent to each calorie multiplied by one t

Tokenize the prompt and pass to LLM.

In [112]:
input_ids = tokenizer(prompt,return_tensors="pt").to("cuda")
#output
output = llm_model.generate(**input_ids,temperature=0.1,do_sample=True,max_new_tokens=50)
decoded_output = tokenizer.decode(output[0])
print(f"Query: {query}")
print(f"RAG answer:\n{decoded_output.replace(prompt, '')}")

Query: What is Osteoporosis, and what causes it?
RAG answer:
<bos>The context does not provide information about Osteoporosis, so I cannot answer this question from the context.<eos>


Now Lets make function to do it.

In [113]:
def ask_query(query:str,
          temperature:int=0.7,
          max_new_token:int=250,
          format_answer_text=True,
          return_answer_only=True):

  ##RETRIEVAL
  #get the score and indices from relevant results
  scores,indices = retrieve_relavent_resources(query=query,embeddings=embeddings)

  #create a list of context retrieval
  retrieve_context = [filtered_chunks[i] for i in indices]

  #add score also in retrieve context
  for i,item in enumerate(retrieve_context):
    item["score"] = scores[i].cpu

  #AUGMENTATION
  prompt = prompt_formatter(query=query,retrieve_context=retrieve_context)


  #GENERATION
  input_ids = tokenizer(prompt,return_tensors="pt").to("cuda")

  output = llm_model.generate(**input_ids,temperature=temperature,do_sample=True,max_new_tokens=max_new_token)

  output = tokenizer.decode(output[0])

  #format the answer
  if format_answer_text:
    #replace prompt and special token
    output=output.replace(prompt,"").replace("<bos>","").replace("<eos>","")

  if return_answer_only:
    return output

In [114]:
query = random.choice(query_list)
print(f"Query:\n{query}")
ask_query(query=query)


Query:
What are the macronutrients, and what roles do they play in the human body?
Query:
What are the macronutrients, and what roles do they play in the human body?


"Sure, here is the answer to the question:\n\nThe macronutrients are carbohydrates, lipids, and proteins. They are all essential for the proper functioning of the human body. They provide energy for all the body's functions, including muscle contraction, nerve transmission, and cell growth."